In [34]:
import requests

In [102]:
from langchain.chat_models import init_chat_model

model = init_chat_model("ollama:qwen2.5:32b")


In [103]:
import asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client
import traceback
from langchain.tools import tool
from typing import List, Dict, Any


# For SSE-based MCP servers
async def tools_sse_mcp_client():
    """Test MCP client over SSE (Server-Sent Events)"""
    print("=== Connecting to SSE MCP Server ===")
    try:
        async with sse_client("http://localhost:8005/mcp") as (read, write):
            async with ClientSession(read, write) as session:
                
                print("✅ Connected to SSE MCP server")
                
                # Initialize the MCP session
                print("\n=== Initializing MCP Session ===")
                init_result = await session.initialize()
                print(f"Initialization: {init_result}")
                
                # List available tools
                print("\n=== Listing Available Tools ===")
                tools_result = await session.list_tools()
                if tools_result.tools:
                    for tool in tools_result.tools:
                        print(f"Tool: {tool.name} - {tool.description}")
                        if hasattr(tool, 'inputSchema'):
                            print(f"  Input schema: {tool.inputSchema}")
                else:
                    print("No tools available")
                
                # List available resources
                print("\n=== Listing Available Resources ===")
                resources_result = await session.list_resources()
                if resources_result.resources:
                    for resource in resources_result.resources:
                        print(f"Resource: {resource.uri} - {resource.name}")
                else:
                    print("No resources available")
                        
    except Exception as e:
        print(f"❌ SSE MCP connection failed: {e}")
        print(f"Full error details:")
        traceback.print_exc()

async def _websearch_mcp_async(user_query: str, fetch_results: bool=True, max_results: int=5):
    """Internal async function for web search via MCP"""
    try:
        async with sse_client("http://localhost:8005/mcp") as (read, write):
            async with ClientSession(read, write) as session:

                print("✅ Connected to SSE MCP server")

                # Initialize the MCP session
                print("\n=== Initializing MCP Session ===")
                init_result = await session.initialize()
                print(f"Initialization: {init_result}")

                if fetch_results:
                    print("\n=== Fetching Web Search Results ===")
                    
                    # Use the correct MCP method to call the tool
                    tool_result = await session.call_tool(
                        name="search_web",
                        arguments={
                            "user_input": user_query,
                            "fetch_content": True,
                            "max_results": max_results
                        }
                    )
                    
                    print(f"Tool call result: {tool_result}")
                    
                    # Extract the actual results from the tool response
                    if tool_result.content:
                        results = []
                        for content_item in tool_result.content:
                            if hasattr(content_item, 'text'):
                                print(f"Search results: {content_item.text}")
                                results.append(content_item.text)
                            else:
                                print(f"Result: {content_item}")
                                results.append(str(content_item))
                        return "\n".join(results)
                    else:
                        print("No results found")
                        return "No results found"

    except Exception as e:
        print(f"❌ SSE MCP connection failed: {e}")
        print(f"Full error details:")
        traceback.print_exc()
        return f"Error: {str(e)}"

@tool
def websearch_sse_mcp_client(user_query: str, max_results: int = 5) -> str:
    """Search the web for information using MCP server.
    
    Args:
        user_query: The search query to execute
        max_results: Maximum number of results to return (default: 5)
    
    Returns:
        Search results as a formatted string
    """
    print(f"Web search initiated with query: {user_query}")
    
    try:
        # Run the async function in a new event loop
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            result = loop.run_until_complete(_websearch_mcp_async(user_query, True, max_results))
            return result
        finally:
            loop.close()
    except Exception as e:
        return f"Error executing web search: {str(e)}"


tools = [websearch_sse_mcp_client]
model_with_tools = model.bind_tools(tools)

In [57]:
query = "Highlight the key features behind gpt5, a web search should answer it. make sure to wait for the tool output"



response = model_with_tools.invoke([{"role": "user", "content": query}])
print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Message content: <think>
Okay, the user is asking about the key features of GPT-5 and wants me to highlight them using a web search. But wait, GPT-5 doesn't exist yet. I remember that OpenAI has released GPT-3.5 and GPT-4, but GPT-5 hasn't been announced or released. So, the user might be confused or maybe they heard a rumor.

First, I should check if there's any official information about GPT-5. Since the user mentioned a web search, I need to use the provided tool to search for it. Let me call the websearch_sse_mcp_client function with the query "GPT-5 key features" to see what comes up.

Wait, but I should make sure that the search is accurate. Maybe the user is mixing up GPT-5 with another model. Let me proceed with the search. The function requires a user_query, so I'll input "GPT-5 key features" and maybe set max_results to 5 as default.

But I should also consider that if there's no information on GPT-5, the search might return nothing or clarify that it's not real. The user mig

In [107]:
sys_prompt = """
You are a ReAct (Reason+Act) agent.
 
GOAL
- Solve the user's task accurately and efficiently.
- Use tools when they reduce uncertainty or effort.
- Keep your internal reasoning private. Do NOT reveal chain-of-thought.
 
TOOLS
You may invoke a websearch

Rules:
- Only call tools that exist in the list above.
- Provide valid inputs matching each tool's schema.
- If a tool errors, extract the signal from the error and try an alternative or explain the limitation.
 
INTERACTION LOOP
- Think privately about whether a tool is needed.
- If NO tool is needed, produce:
 
Final Answer:
<concise, direct answer; include citations if you relied on external sources>
 
- If a tool IS needed, output EXACTLY the following (no extra text):
 
Action: <tool_name>
Action Input:
<valid params for that tool>
 
The system will then return:
 
Observation:
<tool output>
Link to the user question ?

 
Use the observation to decide next steps and, if needed, call another tool.
Stop calling tools once you can answer.
 
STYLE & QUALITY
- Be concise, cite sources when you rely on external factual data.
- State assumptions explicitly if something is uncertain.
- Don't fabricate facts, numbers, APIs, or files.
- Prefer stepwise decomposition internally; present only the necessary result to the user.
 
SAFETY
- Follow all usage restrictions communicated in the tool docs.
- If a request is unsafe or disallowed, refuse briefly and suggest a safe alternative.
 
CONTEXT
Current datetime: {CURRENT_DATETIME}
User locale/timezone: {TIMEZONE}

"""
from datetime import datetime
import time

# Replace placeholders with string values to avoid TypeError from str.replace
sys_prompt = sys_prompt.replace("{CURRENT_DATETIME}", datetime.now().isoformat())  # Placeholder for actual datetime
sys_prompt = sys_prompt.replace("{TIMEZONE}", str(time.tzname[0] if time.tzname else 'UTC'))  # Placeholder for actual timezone


In [108]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent_executor = create_react_agent(model, tools, checkpointer=memory, prompt= sys_prompt)

config = {"configurable": {"thread_id": "12134"}}

In [109]:
input_message = {"role": "user", "content": "but this is not my question"}
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()




================================ Human Message =================================

but this is not my question
================================== Ai Message ==================================

Could you please clarify or provide more details about your question?
